# Linear Regression: Evaluation Metrics

This notebook shows how to check **how good** a Linear Regression model is.

We will learn these metrics:
1. MAE (Mean Absolute Error)
2. MSE (Mean Squared Error)
3. RMSE (Root Mean Squared Error)
4. R² Score
5. Adjusted R² Score
6. MAPE (Mean Absolute Percentage Error)
7. Residual plot (a picture of the errors)

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

## Step 2: Create simple data

We make fake data with 3 input columns (features) and 1 output (target).
You can replace this with your own data later.

In [ ]:
X, y = make_regression(n_samples=200, n_features=3, noise=15, random_state=42)
y = y + 500   # shift values so all are positive (helps MAPE)

df = pd.DataFrame(X, columns=["feature_1", "feature_2", "feature_3"])
df["target"] = y
df.head()

## Step 3: Split data into train and test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

## Step 4: Train the model and make predictions

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred[:5]

## Step 5: Evaluation metrics

### 1. MAE (Mean Absolute Error)
- The **average size of the error**.
- Easy to understand. Same unit as the target.
- Formula: `MAE = average of |actual - predicted|`
- **Lower is better.**

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", round(mae, 4))

### 2. MSE (Mean Squared Error)
- Squares each error, then takes the average.
- Big errors get a **bigger penalty**.
- Formula: `MSE = average of (actual - predicted)²`
- **Lower is better.**

In [ ]:
mse = mean_squared_error(y_test, y_pred)
print("MSE:", round(mse, 4))

### 3. RMSE (Root Mean Squared Error)
- Square root of MSE.
- Same unit as the target, so it is easier to read than MSE.
- Formula: `RMSE = √MSE`
- **Lower is better.**

In [ ]:
rmse = np.sqrt(mse)
print("RMSE:", round(rmse, 4))

### 4. R² Score (R-squared)
- Tells **how much of the change in the target** your model can explain.
- Value is usually between 0 and 1.
  - `1.0` = perfect model
  - `0.0` = model is as bad as guessing the average
  - Below `0` = model is worse than guessing the average
- **Higher is better.**

In [ ]:
r2 = r2_score(y_test, y_pred)
print("R² Score:", round(r2, 4))

### 5. Adjusted R² Score
- R² always goes up when you add more features, even useless ones.
- Adjusted R² fixes this. It **punishes** extra features that do not help.
- Formula: `Adjusted R² = 1 - (1 - R²) × (n - 1) / (n - p - 1)`
  - `n` = number of rows
  - `p` = number of features
- **Higher is better.**

In [ ]:
n = X_test.shape[0]   # number of rows in test data
p = X_test.shape[1]   # number of features

adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R²:", round(adj_r2, 4))

### 6. MAPE (Mean Absolute Percentage Error)
- Error shown as a **percentage**.
- Example: MAPE = 5% means predictions are off by about 5% on average.
- Do not use it when actual values are `0` (it divides by the actual value).
- **Lower is better.**

In [ ]:
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
print("MAPE:", round(mape, 2), "%")

## Step 6: All metrics in one table

In [ ]:
results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²", "Adjusted R²", "MAPE (%)"],
    "Value": [mae, mse, rmse, r2, adj_r2, mape],
    "Better when": ["Lower", "Lower", "Lower", "Higher", "Higher", "Lower"],
})
results["Value"] = results["Value"].round(4)
results

## Step 7: Visual checks

### Actual vs Predicted
If the points are close to the red line, the model is good.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual values")
plt.ylabel("Predicted values")
plt.title("Actual vs Predicted")
plt.show()

### Residual plot
- Residual = `actual - predicted`
- A good model has residuals **scattered randomly around 0**, with no clear pattern.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

## Quick summary

| Metric | What it tells you | Good value |
|---|---|---|
| MAE | Average error | Low |
| MSE | Average squared error (punishes big errors) | Low |
| RMSE | Square root of MSE, same unit as target | Low |
| R² | % of change explained by the model | Close to 1 |
| Adjusted R² | R² that punishes useless features | Close to 1 |
| MAPE | Average error in % | Low |

**Tip:** Never use only one metric. Look at a few of them together, and also look at the plots.